# 01 — EDA: MTSamples Clinical Notes
**Project:** Clinical Medication Extraction | **Phase 1 of the roadmap**

This notebook answers 7 questions. Each one feeds a later phase — nothing here is decoration.
Cells marked **🔨 YOUR TURN** are deliberately incomplete: they're the learning moments. Everything else runs as-is.

**Rule for this notebook:** every chart ends with a one-line markdown interpretation written by you. A chart you can't interpret in one sentence didn't need to exist.

## Setup
Colab: run the pip cell, mount Drive, set `DATA_PATH` to your Drive location.
Local: skip the mount, point `DATA_PATH` at your file.

In [ ]:
# Colab only — quiet installs (~30s). ydata-profiling is optional/heavy, see Q2.
%pip install -q plotly kaleido

In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 120)

# --- paths ---
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    pass

DATA_PATH = '/content/drive/MyDrive/clinical-nlp/data/raw/mtsamples.csv' if IN_COLAB else 'mtsamples.csv'
FIG_DIR   = '/content/drive/MyDrive/clinical-nlp/figures/' if IN_COLAB else 'figures/'
OUT_DIR   = '/content/drive/MyDrive/clinical-nlp/data/working/' if IN_COLAB else 'working/'

import os
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH, index_col=0)
df.columns = [c.strip() for c in df.columns]
print(df.shape)
df.head(3)

## Q1 — Data health
*Feeds: every later phase. Especially train/eval splits in Phases 3–4.*

The usual checks — plus one that matters a lot for scraped data: **duplicates**.

In [ ]:
print('Shape:', df.shape)
print()
print('Nulls per column:')
print(df.isna().sum().to_string())
print()
print('Duplicate transcription texts:', df['transcription'].duplicated().sum())

**⚠️ Big finding — read this.** ~2,600 of 4,999 rows have duplicated transcription text. In this corpus the *same note* is often listed under multiple specialties (e.g., once under *Surgery* and again under *Orthopedic*).

Why it matters: if the same note text lands in both your training data and your gold/eval set, your metrics are inflated and meaningless — the classic **leakage** failure. This is exactly the fan-out-style trap you know from Clarity joins: same entity, multiple rows, silently multiplied.

Decision to make (then log in `decisions.md`):
- For **corpus-level analysis and any modeling**: deduplicate on `transcription` (keep first).
- For **specialty landscape analysis** (Q4): the duplicated view is arguably the honest one (it shows how notes are categorized) — but label your chart accordingly.

We carry two dataframes forward: `df` (raw) and `d` (deduped, null-dropped). Default to `d`.

In [ ]:
d = (df
     .dropna(subset=['transcription'])
     .drop_duplicates(subset='transcription', keep='first')
     .copy())
d['medical_specialty'] = d['medical_specialty'].str.strip()
print(f'Raw rows: {len(df)}  ->  clean unique notes: {len(d)}')

## Q2 — Automated profile (optional, 10 minutes max)
`ydata-profiling` generates a full report in one line. Use it as a *map*, mine it for surprises, then close it — the real EDA is the manual questions below. Skip freely if the install is slow.

In [ ]:
# %pip install -q ydata-profiling
# from ydata_profiling import ProfileReport
# ProfileReport(d, title='MTSamples Profile', minimal=True).to_notebook_iframe()

## Q3 — Length structure
*Feeds: Phase 5 — which notes exceed an LLM's comfortable context? Do we need chunking?*

In [ ]:
d['char_len'] = d['transcription'].str.len()
d['word_len'] = d['transcription'].str.split().str.len()

fig = px.histogram(d, x='word_len', nbins=60,
                   title='Note length distribution (words) — deduped corpus',
                   labels={'word_len': 'words per note'})
fig.add_vline(x=d['word_len'].median(), line_dash='dash',
              annotation_text=f"median {int(d['word_len'].median())}")
fig.write_html(FIG_DIR + 'length_hist.html')
fig.show()

print(d['word_len'].describe().round(0).to_string())

In [ ]:
top10 = d['medical_specialty'].value_counts().head(10).index
fig = px.box(d[d['medical_specialty'].isin(top10)],
             x='medical_specialty', y='word_len',
             hover_data=['sample_name'],   # hover an outlier -> see which note it is
             title='Note length by specialty (top 10) — hover outliers to identify them')
fig.update_xaxes(tickangle=35)
fig.write_html(FIG_DIR + 'length_by_specialty.html')
fig.show()

**🔨 YOUR TURN — interpret (edit this cell):**
- Median note is ~___ words. A 7B instruct model comfortably handles ~___ words of input, so chunking is / is not needed for the typical note.
- Longest note type is ___ ; shortest is ___ .
- One outlier I hovered and read: ___ — it was long/short because ___ .

## Q4 — Specialty landscape
*Feeds: Phase 3 — is the working subset big enough for a stratified gold set?*

In [ ]:
counts = d['medical_specialty'].value_counts()
fig = px.bar(counts.head(15)[::-1], orientation='h',
             title='Unique notes per specialty (top 15, deduped)',
             labels={'value': 'notes', 'index': ''})
fig.update_layout(showlegend=False)
fig.write_html(FIG_DIR + 'specialty_counts.html')
fig.show()

In [ ]:
# The working subset defined in the roadmap:
mask = (d['medical_specialty'].isin(['SOAP / Chart / Progress Notes', 'General Medicine'])
        | d['sample_name'].str.contains('Discharge', case=False, na=False))
work = d[mask].copy()
print('Working subset size:', len(work))
print()
print(work['medical_specialty'].value_counts().head(8).to_string())

**🔨 YOUR TURN — decide (then log in `decisions.md`):**
~600 notes. For a 75-note gold set stratified across these categories, is each stratum big enough? If one category dominates, do you stratify proportionally or equally? (Your RCT sampling instincts apply directly — write 2 sentences.)

## Q5 — Section header census ⭐
*Feeds: Phase 2 — this chart IS the spec for your sectionizer.*

A starter regex is provided. It works — but it has **at least three real failure modes on this corpus**. Your job is to find them (see YOUR TURN below).

In [ ]:
# Starter: ALL-CAPS run (letters, spaces, some punctuation) followed by a colon
header_pat = re.compile(r'([A-Z][A-Z /&()-]{2,40}):')

census = Counter()
for text in d['transcription']:
    census.update(h.strip() for h in header_pat.findall(text))

print(f'Unique header strings found: {len(census)}')
top30 = pd.DataFrame(census.most_common(30), columns=['header', 'count'])

fig = px.bar(top30[::-1], x='count', y='header', orientation='h',
             title='Top 30 section headers across the corpus',
             height=700)
fig.write_html(FIG_DIR + 'header_census.html')
fig.show()

In [ ]:
# Long tail: how many headers appear only once? (Messiness measure)
tail = sum(1 for _, c in census.items() if c == 1)
print(f'{tail} of {len(census)} header strings appear exactly once')
print()
print('Random sample of the weird tail:')
import random
random.seed(7)
print(random.sample([h for h, c in census.items() if c == 1], 15))

**🔨 YOUR TURN — break the regex (the real exercise of this notebook):**
1,800+ unique "headers," most appearing once — that tail is regex over-matching plus genuine dictation chaos. Paste 2–3 full notes into regex101.com with the starter pattern and find its failure modes. Hunt for at least these three:
1. A **false positive**: an ALL-CAPS string matched that is *not* a section header (hint: look at abbreviation-dense lines — HEENT is a header, but what about lab shorthand or 'DR. ABC'?).
2. A **false negative**: a real header the pattern misses (hint: are all headers fully uppercase? any with digits?).
3. The **stray-comma artifact**: headers in this corpus often look like `SUBJECTIVE:,` — does the comma break anything downstream when you split on matches?

Write your improved pattern below and re-run the census. Log the before/after unique-header count in `decisions.md`.

In [ ]:
# 🔨 your improved pattern here:
# header_pat_v2 = re.compile(r'...')
#
# census2 = Counter()
# for text in d['transcription']:
#     census2.update(h.strip() for h in header_pat_v2.findall(text))
# print(len(census), '->', len(census2))

## Q6 — Medication surface check
*Feeds: Phase 3 — are there enough medication mentions in the working subset to be worth annotating?*

Seed list drawn from drugs you know cold (QT project + common meds). Crude substring matching is fine at EDA stage — precision comes in Phase 2.

In [ ]:
seed_drugs = ['amiodarone', 'sotalol', 'metoprolol', 'atenolol', 'digoxin',
              'warfarin', 'aspirin', 'lisinopril', 'furosemide', 'atorvastatin',
              'insulin', 'metformin', 'prednisone', 'albuterol', 'omeprazole',
              'gabapentin', 'oxycodone', 'acetaminophen', 'ibuprofen', 'azithromycin']

lower = d.assign(txt=d['transcription'].str.lower())
rows = []
for drug in seed_drugs:
    hits = lower[lower['txt'].str.contains(drug, regex=False)]
    for spec, n in hits['medical_specialty'].value_counts().head(20).items():
        rows.append({'drug': drug, 'specialty': spec, 'notes_with_mention': n})

heat = (pd.DataFrame(rows)
        .pivot_table(index='drug', columns='specialty',
                     values='notes_with_mention', fill_value=0))
keep_cols = heat.sum().sort_values(ascending=False).head(10).index
fig = px.imshow(heat[keep_cols], aspect='auto',
                title='Notes mentioning each seed drug, by specialty (top 10 specialties)',
                labels={'color': 'notes'}, height=600)
fig.write_html(FIG_DIR + 'drug_heatmap.html')
fig.show()

in_work = work['transcription'].str.lower()
n_with_med = in_work.apply(lambda t: any(dr in t for dr in seed_drugs)).sum()
print(f'Working-subset notes mentioning >=1 seed drug: {n_with_med} / {len(work)}')

**🔨 YOUR TURN — interpret:** Is that hit rate high enough that a random 75-note gold sample will contain plenty of medication events, or should Phase 3 sample *conditioned on* containing a drug mention? (Careful — conditioning changes what your metrics generalize to. One sentence on the tradeoff; you've navigated exactly this in cohort definitions.)

## Q7 — Read 10 notes (no tooling)
The highest-value cell in the notebook. Read them fully. Then fill in the observation cell.

In [ ]:
sample_notes = work.sample(10, random_state=42)
for i, (_, row) in enumerate(sample_notes.iterrows(), 1):
    print('=' * 90)
    print(f"NOTE {i} | {row['medical_specialty']} | {row['sample_name']}")
    print('=' * 90)
    print(row['transcription'])
    print()

**🔨 YOUR TURN — observations (edit; aim for 8–10 bullets):**
- Formatting quirks I noticed: ...
- How medications are actually written (dose/route/frequency phrasing): ...
- Negations and historical mentions I spotted ("denies", "discontinued", "was previously on"): ...
- Things my sectionizer will definitely trip on: ...
- Anything that surprised me: ...

## Save the working subset & wrap up

In [ ]:
work.to_parquet(OUT_DIR + 'notes_subset.parquet')
print('Saved:', OUT_DIR + 'notes_subset.parquet', '| rows:', len(work))

### Before you close this session
1. Fill every 🔨 cell — they're the notebook's actual value.
2. `decisions.md`: log (a) dedup decision, (b) gold-set stratification plan, (c) header regex v1→v2 result, (d) conditioned-vs-random sampling call.
3. Commit to GitHub: *File → Save a copy in GitHub*.
4. `learning-log.md`: one line.

**Next:** `02_sectionizer.ipynb` — turn the Q5 header census into `split_sections()`. The roadmap has the spec.